In [1]:
from qldpc_sim import *
from qldpc_sim.qldpc_experiment import *
from qldpc_sim.qec_code import *
from qldpc_sim.data_structure import *
import stim
import numpy as np

In [2]:
from qldpc_sim.qec_code.rotated_surface_code import RotatedSurfaceCode


def setup_cnot_exp():
    patch1 = RotatedSurfaceCode.from_distance(3, code_name="p1")
    patch2 = RotatedSurfaceCode.from_distance(3, code_name="p2")
    qm = qldpc_experiment.QuantumMemory(size=600)
    patches = [patch1, patch2]
    mapqb = {}
    for c in patches:
        for lq in c.logical_qubits:
            mapqb[lq.logical_x] = c
            mapqb[lq.logical_z] = c

    ctx = Context(
        codes=[patch1, patch2],
        logical_qubits=patch1.logical_qubits + patch2.logical_qubits,
        initial_assignement=mapqb,
        memory=qm,
    )

    return ctx

In [3]:
from qldpc_sim.ckbb_surgery.measurement import CKBBMeasurement
from qldpc_sim.rsc_surgery.rsc_surgery import SurgeMeasurement
from qldpc_sim.qldpc_experiment.qec_gadget import Readout


def get_joinm_program(initial_state1, initial_state2, joint_pauli, readout_basis):
    context = setup_cnot_exp()

    p1 = context.codes[0]
    p2 = context.codes[1]
    if joint_pauli == "XX":
        ckbbm = [
            CKBBMeasurement(
                distance=2,
                context=context,
                tag="CKBBM",
                logical_targets=[
                    p2.logical_qubits[0].logical_x,
                    p1.logical_qubits[0].logical_x,
                ],
            ),
        ]
    else:
        ckbbm = [
            CKBBMeasurement(
                distance=2,
                context=context,
                tag="CKBBM",
                logical_targets=[
                    p2.logical_qubits[0].logical_z,
                    p1.logical_qubits[0].logical_z,
                ],
            ),
        ]
    return (
        [
            InitializeCode(
                code=p1,
                context=context,
                tag=f"init_{p1.id}",
                initial_state=initial_state1,
            )
        ]
        + [
            InitializeCode(
                code=p2,
                context=context,
                tag=f"init_{p2.id}",
                initial_state=initial_state2,
            )
        ]
        + [
            StabMeasurement(code=p, context=context, tag=f"isb_2_{p.id}", round=3)
            for p in [p1, p2]
        ]
        + ckbbm
        + [
            StabMeasurement(code=p, context=context, tag=f"isb_3_{p.id}", round=3)
            for p in [p1, p2]
        ]
        + [
            LM(
                logical_targets=[
                    (
                        p1.logical_qubits[0].logical_x
                        if readout_basis == "X"
                        else p1.logical_qubits[0].logical_z
                    )
                ],
                context=context,
                tag=f"p1",
                basis=PauliChar.X if readout_basis == "X" else PauliChar.Z,
                reset_qubits=False,
            ),
            LM(
                logical_targets=[
                    (
                        p2.logical_qubits[0].logical_x
                        if readout_basis == "X"
                        else p2.logical_qubits[0].logical_z
                    )
                ],
                context=context,
                tag=f"p2",
                basis=PauliChar.X if readout_basis == "X" else PauliChar.Z,
                reset_qubits=False,
            ),
            Readout(
                code=p1,
                context=context,
                tag=f"readout_{p1.id}",
                basis=PauliChar.X if readout_basis == "X" else PauliChar.Z,
            ),
            Readout(
                code=p2,
                context=context,
                tag=f"readout_{p2.id}",
                basis=PauliChar.X if readout_basis == "X" else PauliChar.Z,
            ),
        ]
    ), context

In [4]:
from collections import Counter
from itertools import combinations
import json
from pathlib import Path

from qldpc_sim.qldpc_experiment.interpreter import concat_events_per_sample, run, xor_event_nodes

prog, ctx = get_joinm_program(
    initial_state1=PauliEigenState.X_plus,
    initial_state2=PauliEigenState.X_plus,
    joint_pauli="ZZ",
    readout_basis="X",
)

sample_outcome = run(context=ctx, program=prog, num_samples=100)
print("Available measurement keys:")
for key in sample_outcome.keys():
    print(f"  {key}")

mxx_r0 = xor_event_nodes(
    sample_outcome, "ckbb_ancila_var_readout_observable_parity_outcome"
)

p1_corr = None
p2_corr = None
for k in sample_outcome.keys():
    if "ckbb_ancila_var_readout_observable_corr_" in k:
        print(f"Found correction key: {k}")
        if "p1" in k:
            print(f"Extracting p1 correction from key: {k}")
            p1_corr = xor_event_nodes(sample_outcome, k)
        elif "p2" in k:
            print(f"Extracting p2 correction from key: {k}")
            p2_corr = xor_event_nodes(sample_outcome, k)

p1 = xor_event_nodes(sample_outcome, "p1", tag=None)
p2 = xor_event_nodes(sample_outcome, "p2", tag=None)
# cps = concat_events_per_sample({**p1, **p1_corr, **p2, **p2_corr, **mxx_r0})
# outcome_counter = Counter(tuple(v) for v in cps.values())
# print("Number of possible outcomes:", len(outcome_counter))
# print("Outcome distribution:", outcome_counter)

UnboundLocalError: cannot access local variable 'full_ancilla_tanner' where it is not associated with a value

In [ ]:
# Build corrected outcomes robustly even if p1_corr / p2_corr are None or empty

def _extract_series(event_dict, fallback_len):
    if not event_dict:
        return [1] * fallback_len
    first = next(iter(event_dict.values()), None)
    return first if first is not None else [1] * fallback_len

p1_key = next(iter(p1.keys()))
p2_key = next(iter(p2.keys()))

p1_vals = p1[p1_key]
p2_vals = p2[p2_key]

p1_corr_vals = _extract_series(p1_corr, len(p1_vals))
p2_corr_vals = _extract_series(p2_corr, len(p2_vals))

p1_corrected = [a * b for a, b in zip(p1_vals, p1_corr_vals)]
p2_corrected = [a * b for a, b in zip(p2_vals, p2_corr_vals)]

corrected_cps = concat_events_per_sample(
    {
        "p1": p1_corrected,
        "p2": p2_corrected,
        **mxx_r0,
    }
)
corrected_outcome_counter = Counter(tuple(v) for v in corrected_cps.values())

print("Number of corrected possible outcomes:", len(corrected_outcome_counter))
print("Corrected outcome distribution:", corrected_outcome_counter)


Number of corrected possible outcomes: 8
Corrected outcome distribution: Counter({(1, 1, 1): 18, (-1, -1, 1): 14, (1, 1, -1): 14, (1, -1, -1): 13, (-1, 1, 1): 12, (-1, -1, -1): 11, (-1, 1, -1): 10, (1, -1, 1): 8})
